# M3T3 - Productor de eventos para streaming

Este cuaderno genera eventos JSON para simular flujos de clickstream y transacciones. Los eventos se escriben como JSON Lines en `input/m3t3_events`, de forma que Spark Structured Streaming pueda leerlos como una fuente continua de ficheros.

Ejecuta este productor en paralelo con `M3T3_consumidor_pipeline_streaming.ipynb`.

In [ ]:
import os
import json
import time
import random
from datetime import datetime, timezone, timedelta

INPUT_DIR = os.path.abspath("input/m3t3_events")
os.makedirs(INPUT_DIR, exist_ok=True)

print("Directorio de eventos:", INPUT_DIR)

## Configuración del flujo

Modifica estos parámetros para cambiar volumen, frecuencia o probabilidad de eventos tardíos.

In [ ]:
USERS = [f"u{i:03d}" for i in range(1, 21)]
URLS = ["/", "/pricing", "/docs", "/checkout", "/account", "/search"]
CARDS = [f"card-{i:03d}" for i in range(1, 11)]
MERCHANTS = ["shop_01", "shop_02", "travel_01", "games_01", "market_01"]

EVENTS_PER_BATCH = 12
SLEEP_SECONDS = 5
MAX_ITERATIONS = 24
LATE_EVENT_PROBABILITY = 0.15

In [ ]:
def event_timestamp():
    now = datetime.now(timezone.utc)
    if random.random() < LATE_EVENT_PROBABILITY:
        now = now - timedelta(minutes=random.randint(3, 12))
    return now.strftime("%Y-%m-%dT%H:%M:%SZ")

def crear_click(iteration, position):
    return {
        "event_id": f"clk-{int(time.time())}-{iteration}-{position}",
        "event_type": "click",
        "ts": event_timestamp(),
        "userId": random.choice(USERS),
        "url": random.choice(URLS),
        "amount": None,
        "cardId": None,
        "merchant": None,
        "source": "m3t3_clickstream_simulator"
    }

def crear_transaccion(iteration, position):
    return {
        "event_id": f"tx-{int(time.time())}-{iteration}-{position}",
        "event_type": "transaction",
        "ts": event_timestamp(),
        "userId": random.choice(USERS),
        "url": None,
        "amount": round(random.uniform(5, 500), 2),
        "cardId": random.choice(CARDS),
        "merchant": random.choice(MERCHANTS),
        "source": "m3t3_transaction_simulator"
    }

crear_click(1, 1), crear_transaccion(1, 2)

## Generación continua

Cada iteración escribe un fichero nuevo con eventos mezclados. El consumidor detectará esos ficheros como nuevos microbatches.

In [ ]:
total = 0

for i in range(MAX_ITERATIONS):
    filename = os.path.join(INPUT_DIR, f"events_{int(time.time())}_{i}.json")
    events = []
    for j in range(EVENTS_PER_BATCH):
        if random.random() < 0.65:
            events.append(crear_click(i + 1, j + 1))
        else:
            events.append(crear_transaccion(i + 1, j + 1))

    with open(filename, "w", encoding="utf-8") as file:
        for event in events:
            file.write(json.dumps(event, ensure_ascii=False) + "\n")

    total += len(events)
    print(f"[{i + 1}] {len(events)} eventos escritos en {filename}. Total: {total}")
    time.sleep(SLEEP_SECONDS)